# Carga y Preparación de Datos con MedMNIST (`PneumoniaMNIST`)

En este cuaderno se documenta paso a paso cómo funcionan los dos archivos encargados de la descarga, transformación y carga de datos biomédicos en el proyecto:

1. **`src/data/datamodule.py` (`MedDataModule`)**: Clase modular que descarga el dataset, transforma las imágenes a tensores en el rango $[-1, 1]$ y crea los `DataLoader` de PyTorch.
2. **`scripts/download_data.py`**: Script de utilidad para descargar los datos en local y verificar que las dimensiones y rangos de los tensores son correctos.

---
## 1. Configuración de Rutas e Importaciones

Tanto en `download_data.py` como en nuestros cuadernos, primero añadimos la carpeta `qml-biomedico` al `sys.path` de Python para poder importar los módulos propios desde `src/`.

In [1]:
import sys
from pathlib import Path
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
import medmnist
from medmnist import INFO

# Apuntar a la carpeta qml-biomedico del proyecto
PROJECT_ROOT = Path.cwd().parent.parent / "qml-biomedico"
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path("qml-biomedico").resolve()

sys.path.append(str(PROJECT_ROOT))
DATA_DIR = PROJECT_ROOT / "data"

print(f"Raíz del código: {PROJECT_ROOT}")
print(f"Directorio de datos: {DATA_DIR}")

Raíz del código: C:\Users\Gonzalo\Documents\TFG\Quantum-MachineLearning-AnalysisBiomedical\qml-biomedico
Directorio de datos: C:\Users\Gonzalo\Documents\TFG\Quantum-MachineLearning-AnalysisBiomedical\qml-biomedico\data


---
## 2. ¿Cómo funciona `MedDataModule` (`src/data/datamodule.py`)?

Vamos a desglosar cada bloque de la clase `MedDataModule`.

### 2.1. Obtención de metadatos de MedMNIST
La librería `medmnist` incluye un diccionario global llamado `INFO`. Al pasarle el identificador `"pneumoniamnist"`, nos devuelve la información del dataset (qué clase de Python usar, qué significan las etiquetas `0` y `1`, cuántos canales tiene la imagen, etc.):

In [2]:
flag = "pneumoniamnist"
info = INFO[flag]

# getattr busca dentro del paquete medmnist la clase cuyo nombre indica info["python_class"]
DataClass = getattr(medmnist, info["python_class"])

print(f"Clase de Python en medmnist: {info['python_class']}")
print(f"Descripción de tarea: {info['task']}")
print(f"Significado de etiquetas: {info['label']}")
print(f"Número de canales: {info['n_channels']} (Escala de grises)")
print(f"Número de muestras por split: {info['n_samples']}")

Clase de Python en medmnist: PneumoniaMNIST
Descripción de tarea: binary-class
Significado de etiquetas: {'0': 'normal', '1': 'pneumonia'}
Número de canales: 1 (Escala de grises)
Número de muestras por split: {'train': 4708, 'val': 524, 'test': 624}


### 2.2. El Pipeline de Transformaciones (`transforms.Compose`)

Las imágenes originales guardadas en `pneumoniamnist.npz` son matrices de $28 \times 28$ píxeles con valores enteros entre `0` (negro) y `255` (blanco).

En `MedDataModule` definimos dos transformaciones encadenadas:
1. `transforms.ToTensor()`: Convierte la imagen a un tensor de PyTorch de forma `(1, 28, 28)` y divide cada píxel entre $255$, dejándolo en el rango $[0.0, 1.0]$.
2. `transforms.Normalize(mean=[0.5], std=[0.5])`: Aplica la fórmula:
   $$x_{\text{norm}} = \frac{x - 0.5}{0.5} = 2x - 1$$
   Así, un píxel con valor $0.0$ pasa a valer $-1.0$, y un píxel con valor $1.0$ pasa a valer $+1.0$. El rango final queda en **$[-1.0, 1.0]$**.

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

print("Transformaciones configuradas:")
print(transform)

Transformaciones configuradas:
Compose(
    ToTensor()
    Normalize(mean=[0.5], std=[0.5])
)


### 2.3. Código Completo de `MedDataModule` (`src/data/datamodule.py`)

Uniendo la inicialización, el método `setup()` (que descarga los splits `train`, `val` y `test` si no están en la carpeta `data/`) y los tres métodos `DataLoader` (que entregan los datos en lotes o *batches*), tenemos nuestra clase completa:

In [4]:
from src.data.datamodule import MedDataModule

print("Módulo MedDataModule importado correctamente desde src.data.datamodule")

Módulo MedDataModule importado correctamente desde src.data.datamodule


 Extracto de referencia de `src/data/datamodule.py`:

```python
class MedDataModule:
    def __init__(self, flag="pneumoniamnist", batch_size=32, data_dir="data", num_workers=2):
        self.flag = flag.lower()
        self.batch_size = batch_size
        self.data_dir = Path(data_dir)
        self.num_workers = num_workers

        self.info = INFO[self.flag]
        self.DataClass = getattr(medmnist, self.info["python_class"])

        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.5])
        ])

    def setup(self):
        self.data_dir.mkdir(parents=True, exist_ok=True)
        self.train_dataset = self.DataClass(split="train", transform=self.transform, download=True, root=str(self.data_dir))
        self.val_dataset   = self.DataClass(split="val",   transform=self.transform, download=True, root=str(self.data_dir))
        self.test_dataset  = self.DataClass(split="test",  transform=self.transform, download=True, root=str(self.data_dir))
```

---
## 3. ¿Qué hace `scripts/download_data.py`?

El script `download_data.py` instancia `MedDataModule`, llama a `setup()` para asegurar que el archivo `pneumoniamnist.npz` esté descargado en local y extrae **un único lote (*batch*) de 16 imágenes** usando `next(iter(loader))` para verificar que los tensores tienen la forma y el rango esperados:

In [5]:
print("Iniciando descarga y validación de MedMNIST (PneumoniaMNIST)...")
dm = MedDataModule(flag="pneumoniamnist", batch_size=16, data_dir=str(DATA_DIR), num_workers=0)
dm.setup()

# Obtenemos el DataLoader de entrenamiento y extraemos el primer batch
loader = dm.train_dataloader()
images, labels = next(iter(loader))

print("\n--- Verificación de Tensores ---")
print(f"Batch Shape: {images.shape}")   # [16 imágenes, 1 canal, 28 alto, 28 ancho]
print(f"Labels Shape: {labels.shape}")  # [16 etiquetas, 1 columna (0=normal, 1=neumonía)]
print(f"Rango de valores: min={images.min():.2f}, max={images.max():.2f}")
print(f"Total imágenes en train: {len(dm.train_dataset)} | val: {len(dm.val_dataset)} | test: {len(dm.test_dataset)}")

Iniciando descarga y validación de MedMNIST (PneumoniaMNIST)...

--- Verificación de Tensores ---
Batch Shape: torch.Size([16, 1, 28, 28])
Labels Shape: torch.Size([16, 1])
Rango de valores: min=-1.00, max=0.91
Total imágenes en train: 4708 | val: 524 | test: 624


### Interpretación de la salida:
- **`Batch Shape: torch.Size([16, 1, 28, 28])`**:
  - `16`: Tamaño del lote (`batch_size=16`).
  - `1`: Un solo canal de color (radiografía en escala de grises).
  - `28, 28`: Alto y ancho en píxeles.
- **`Labels Shape: torch.Size([16, 1])`**:
  - Un vector columna con `0` (Normal) o `1` (Neumonía) para cada una de las 16 radiografías del lote.
- **`min=-1.00, max=0.91`**:
  - Confirma que los píxeles ya están normalizados dentro del intervalo $[-1.0, 1.0]$.